In [64]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

DATA_PATH = Path("D:/PENS-EEPIS/SDT A Semester 4 2026/Project TA/dataset/engineered/data_engineeredEDA.csv")
OUTPUT_DIR = Path("D:/PENS-EEPIS/SDT A Semester 4 2026/Project TA/artifacts/metrics/supplier_selection_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

### 1. Setup Library
Cell ini menyiapkan library utama dan folder output. Notebook ini memakai dataset engineered yang sudah ada, bukan data dummy.

In [65]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.info()

Shape: (180519, 73)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 73 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Id                    180519 non-null  int64  
 12  Customer S

### 2. Load Dataset
Dataset yang digunakan adalah `dataset/engineered/data_engineeredEDA.csv` dengan 180.519 transaksi dan 73 kolom. Karena dataset tidak memiliki kolom supplier asli, notebook ini memakai `Product Card Id` dan `Product Name` sebagai proxy kandidat supplier, lalu ranking dilakukan di dalam masing-masing kategori produk.

In [66]:
df_work = df.copy()

df_work["is_consumer_segment"] = (df_work["Customer Segment"].str.lower() == "consumer").astype(int)
df_work["order_row"] = 1

group_cols = [
    "Category Id",
    "Category Name",
    "Product Card Id",
    "Product Name",
]

supplier_df = (
    df_work.groupby(group_cols)
    .agg(
        total_transactions=("order_row", "sum"),
        total_orders=("Order Id", "nunique"),
        total_quantity=("Order Item Quantity", "sum"),
        total_sales=("Sales", "sum"),
        total_profit=("Order Profit Per Order", "sum"),
        gross_purchase_cost=("expected_gross_sales", "sum"),
        total_discount=("Order Item Discount", "sum"),
        total_item_gap=("abs_item_total_gap", "sum"),
        avg_product_price=("Product Price", "mean"),
        avg_discount_rate=("Order Item Discount Rate", "mean"),
        avg_profit_margin=("profit_margin", "mean"),
        avg_benefit_margin=("benefit_margin", "mean"),
        late_rate=("is_late", "mean"),
        severe_delay_rate=("severe_delay", "mean"),
        avg_actual_delay=("actual_delay", "mean"),
        avg_shipping_speed_ratio=("shipping_speed_ratio", "mean"),
        price_inconsistency_rate=("has_price_inconsistency", "mean"),
        country_mismatch_rate=("country_mismatch", "mean"),
        state_mismatch_rate=("state_mismatch", "mean"),
        city_mismatch_rate=("city_mismatch", "mean"),
        inactive_rate=("is_product_inactive", "mean"),
        high_discount_rate=("high_discount_flag", "mean"),
        negative_profit_rate=("negative_profit_flag", "mean"),
        negative_margin_rate=("negative_margin_flag", "mean"),
        consumer_order_rate=("is_consumer_segment", "mean"),
        market_count=("Market", "nunique"),
        order_country_count=("Order Country", "nunique"),
        customer_country_count=("Customer Country", "nunique"),
    )
    .reset_index()
    .rename(
        columns={
            "Category Id": "category_id",
            "Category Name": "category_name",
            "Product Card Id": "candidate_id",
            "Product Name": "candidate_name",
        }
    )
)

supplier_df["category_total_sales"] = supplier_df.groupby("category_name")["total_sales"].transform("sum")
supplier_df["category_total_orders"] = supplier_df.groupby("category_name")["total_orders"].transform("sum")
supplier_df["category_sales_share"] = supplier_df["total_sales"] / supplier_df["category_total_sales"]
supplier_df["category_order_share"] = supplier_df["total_orders"] / supplier_df["category_total_orders"]

supplier_df[
    [
        "category_name",
        "candidate_id",
        "candidate_name",
        "total_orders",
        "total_sales",
        "category_sales_share",
        "consumer_order_rate",
    ]
].head()

,category_name,candidate_id,candidate_name,total_orders,total_sales,category_sales_share,consumer_order_rate
0,Soccer,19,Nike Men's Fingertrap Max Training Shoe,63,7999.359866,0.302124,0.500000
1,Soccer,24,Elevation Training Mask 2.0,74,18477.689972,0.697876,0.581081
2,Baseball & Softball,35,adidas Brazuca 2014 Official Match Ball,65,10399.350357,0.110564,0.507692
3,Baseball & Softball,37,adidas Kids' F5 Messi FG Soccer Cleat,262,27327.190540,0.290538,0.530534
4,Baseball & Softball,44,adidas Men's F10 Messi TRX FG Soccer Cleat,303,56330.611645,0.598898,0.540984


### 3. Agregasi Kandidat Berdasarkan Kategori
Cell ini membuat alternatif pemilihan per kategori. `category_name` dipakai sebagai section, sedangkan `candidate_id` dan `candidate_name` adalah kandidat supplier/proxy product di dalam kategori tersebut. Dengan cara ini, produk di kategori berbeda tidak dibandingkan secara langsung karena kebutuhan konsumen dan karakter barangnya berbeda.

In [67]:
def category_minmax_score(df, column, score_type="benefit", group_col="category_name"):
    def transform(series):
        min_value = series.min()
        max_value = series.max()
        if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
            return pd.Series(3.0, index=series.index)

        if score_type == "benefit":
            return 1 + 4 * ((series - min_value) / (max_value - min_value))

        return 1 + 4 * ((max_value - series) / (max_value - min_value))

    return df.groupby(group_col)[column].transform(transform)


def risk_level(score):
    if score >= 4.0:
        return "Low"
    if score >= 3.0:
        return "Medium"
    if score >= 2.0:
        return "High"
    return "Critical"

### 4. Helper Normalisasi Per Kategori
Normalisasi dilakukan per kategori, bukan global. Ini penting agar kandidat di kategori besar tidak otomatis terlihat lebih baik hanya karena volumenya lebih besar. Skor dibuat pada skala 1 sampai 5.

In [68]:
order_threshold = supplier_df.groupby("category_name")["total_orders"].transform(
    lambda series: max(5, series.quantile(0.25))
)

supplier_df["prequalified"] = (
    (supplier_df["total_orders"] >= order_threshold)
    & (supplier_df["total_sales"] > 0)
    & (supplier_df["total_quantity"] > 0)
)

supplier_df["prequalification_note"] = np.where(
    supplier_df["prequalified"],
    "Passed",
    "Failed: transaksi kategori terlalu rendah atau data sales/quantity tidak valid",
)

supplier_df["compliance_passed"] = (
    (supplier_df["price_inconsistency_rate"] <= 0.10)
    & (supplier_df["negative_margin_rate"] <= 0.85)
)

supplier_df["compliance_note"] = np.where(
    supplier_df["compliance_passed"],
    "Passed",
    "Failed: price inconsistency atau negative margin terlalu tinggi",
)

supplier_df[
    [
        "category_name",
        "candidate_name",
        "total_orders",
        "prequalified",
        "prequalification_note",
        "compliance_passed",
        "compliance_note",
    ]
].head(10)

,category_name,candidate_name,total_orders,prequalified,prequalification_note,compliance_passed,compliance_note
0,Soccer,Nike Men's Fingertrap Max Training Shoe,63,False,Failed: transaksi kategori terlalu rendah atau...,True,Passed
1,Soccer,Elevation Training Mask 2.0,74,True,Passed,True,Passed
2,Baseball & Softball,adidas Brazuca 2014 Official Match Ball,65,False,Failed: transaksi kategori terlalu rendah atau...,True,Passed
3,Baseball & Softball,adidas Kids' F5 Messi FG Soccer Cleat,262,True,Passed,True,Passed
4,Baseball & Softball,adidas Men's F10 Messi TRX FG Soccer Cleat,303,True,Passed,True,Passed
5,Basketball,Diamondback Boys' Insight 24 Performance Hybr,28,True,Passed,True,Passed
6,Basketball,SOLE E25 Elliptical,10,False,Failed: transaksi kategori terlalu rendah atau...,True,Passed
7,Basketball,Diamondback Girls' Clarity 24 Hybrid Bike 201,27,True,Passed,True,Passed
8,Lacrosse,Nike Kids' Grade School KD VI Basketball Shoe,62,False,Failed: transaksi kategori terlalu rendah atau...,True,Passed
9,Lacrosse,Under Armour Men's Tech II T-Shirt,279,True,Passed,True,Passed


### 5. Pre-Qualification dan Compliance Screening
Karena dataset tidak memiliki dokumen legal supplier, pre-qualification diproksikan dengan volume transaksi minimum per kategori. Compliance diproksikan dengan indikator data dan transaksi seperti price inconsistency, negative margin, dan inactive product.

In [69]:
supplier_df["financial_risk_score"] = (
    0.45 * category_minmax_score(supplier_df, "avg_profit_margin", "benefit")
    + 0.35 * category_minmax_score(supplier_df, "total_profit", "benefit")
    + 0.20 * category_minmax_score(supplier_df, "negative_profit_rate", "cost")
)

supplier_df["delivery_risk_score"] = (
    0.45 * category_minmax_score(supplier_df, "late_rate", "cost")
    + 0.35 * category_minmax_score(supplier_df, "avg_actual_delay", "cost")
    + 0.20 * category_minmax_score(supplier_df, "severe_delay_rate", "cost")
)

supplier_df["quality_risk_score"] = (
    0.40 * category_minmax_score(supplier_df, "price_inconsistency_rate", "cost")
    + 0.35 * category_minmax_score(supplier_df, "negative_margin_rate", "cost")
    + 0.25 * category_minmax_score(supplier_df, "inactive_rate", "cost")
)

supplier_df["supply_disruption_risk_score"] = (
    0.35 * category_minmax_score(supplier_df, "severe_delay_rate", "cost")
    + 0.30 * category_minmax_score(supplier_df, "late_rate", "cost")
    + 0.20 * category_minmax_score(supplier_df, "total_orders", "benefit")
    + 0.15 * category_minmax_score(supplier_df, "market_count", "benefit")
)

supplier_df["geographical_risk_score"] = (
    0.40 * category_minmax_score(supplier_df, "country_mismatch_rate", "cost")
    + 0.30 * category_minmax_score(supplier_df, "state_mismatch_rate", "cost")
    + 0.30 * category_minmax_score(supplier_df, "city_mismatch_rate", "cost")
)

supplier_df["compliance_risk_score"] = (
    0.45 * category_minmax_score(supplier_df, "price_inconsistency_rate", "cost")
    + 0.30 * category_minmax_score(supplier_df, "high_discount_rate", "cost")
    + 0.25 * category_minmax_score(supplier_df, "inactive_rate", "cost")
)

supplier_df["consumer_fit_score"] = (
    0.60 * category_minmax_score(supplier_df, "consumer_order_rate", "benefit")
    + 0.40 * category_minmax_score(supplier_df, "category_order_share", "benefit")
)

supplier_df["cyber_data_risk_score"] = 3.0

risk_weights = {
    "financial_risk_score": 0.18,
    "delivery_risk_score": 0.20,
    "quality_risk_score": 0.20,
    "supply_disruption_risk_score": 0.14,
    "geographical_risk_score": 0.08,
    "compliance_risk_score": 0.10,
    "consumer_fit_score": 0.05,
    "cyber_data_risk_score": 0.05,
}

supplier_df["risk_score"] = sum(
    supplier_df[column] * weight for column, weight in risk_weights.items()
).round(4)

supplier_df["risk_level"] = supplier_df["risk_score"].apply(risk_level)

supplier_df[
    [
        "category_name",
        "candidate_name",
        "financial_risk_score",
        "delivery_risk_score",
        "quality_risk_score",
        "consumer_fit_score",
        "risk_score",
        "risk_level",
    ]
].sort_values(["category_name", "risk_score"], ascending=[True, False]).head(15)

,category_name,candidate_name,financial_risk_score,delivery_risk_score,quality_risk_score,consumer_fit_score,risk_score,risk_level
84,Accessories,Team Golf Pittsburgh Steelers Putter Grip,4.920921,5.000000,3.918104,3.043999,4.0761,Low
86,Accessories,Team Golf Texas Longhorns Putter Grip,3.941028,3.601405,3.519267,3.365623,3.4931,Medium
83,Accessories,Team Golf San Francisco Giants Putter Grip,2.897089,3.119545,3.046464,2.957971,2.9726,High
85,Accessories,Team Golf New England Patriots Putter Grip,1.647073,4.041592,2.355063,3.400000,2.8078,High
87,Accessories,Team Golf Tennessee Volunteers Putter Grip,1.725631,3.549646,1.748454,1.950000,2.5594,High
82,Accessories,Team Golf St. Louis Cardinals Putter Grip,1.933813,1.452396,2.809543,4.245661,2.5053,High
28,As Seen on TV!,Nike Men's Free TR 5.0 TB Training Shoe,3.000000,3.000000,3.000000,3.000000,3.0000,Medium
101,Baby,Baby sweater,3.000000,3.000000,3.000000,3.000000,3.0000,Medium
2,Baseball & Softball,adidas Brazuca 2014 Official Match Ball,3.600000,4.074444,4.500000,1.000000,3.6105,Medium
4,Baseball & Softball,adidas Men's F10 Messi TRX FG Soccer Cleat,3.047721,3.683368,1.500000,5.000000,2.9393,High


### 6. Risk Scorecard Per Kategori
Risk Scorecard dihitung dari indikator yang tersedia di dataset. Financial risk, delivery risk, quality risk, supply disruption risk, geographical risk, compliance risk, dan consumer fit dihitung secara relatif di dalam kategori masing-masing.

In [70]:
supplier_df["delay_penalty"] = (
    supplier_df["total_sales"]
    * supplier_df["late_rate"]
    * supplier_df["avg_actual_delay"].clip(lower=0)
    * 0.01
)

supplier_df["quality_penalty"] = (
    supplier_df["total_item_gap"]
    + supplier_df["total_sales"] * supplier_df["price_inconsistency_rate"] * 0.02
    + supplier_df["total_sales"] * supplier_df["negative_margin_rate"] * 0.02
)

supplier_df["discount_penalty"] = supplier_df["total_discount"] * 0.10

supplier_df["tco"] = (
    supplier_df["gross_purchase_cost"]
    + supplier_df["delay_penalty"]
    + supplier_df["quality_penalty"]
    + supplier_df["discount_penalty"]
).round(2)

supplier_df[
    [
        "category_name",
        "candidate_name",
        "gross_purchase_cost",
        "delay_penalty",
        "quality_penalty",
        "discount_penalty",
        "tco",
    ]
].sort_values(["category_name", "tco"]).head(15)

,category_name,candidate_name,gross_purchase_cost,delay_penalty,quality_penalty,discount_penalty,tco
85,Accessories,Team Golf New England Patriots Putter Grip,20566.769811,64.595926,87.889806,216.515999,20935.77
83,Accessories,Team Golf San Francisco Giants Putter Grip,21766.289800,76.392877,90.991548,221.474999,22155.15
84,Accessories,Team Golf Pittsburgh Steelers Putter Grip,22166.129796,61.731494,72.194066,240.313998,22540.37
86,Accessories,Team Golf Texas Longhorns Putter Grip,22366.049794,75.278178,86.821502,242.493998,22770.64
87,Accessories,Team Golf Tennessee Volunteers Putter Grip,22865.849790,77.362792,96.116762,242.503998,23281.83
82,Accessories,Team Golf St. Louis Cardinals Putter Grip,23940.419780,114.158627,99.493296,232.628998,24386.70
28,As Seen on TV!,Nike Men's Free TR 5.0 TB Training Shoe,20597.939559,80.182292,109.088293,207.837000,20995.05
101,Baby,Baby sweater,12229.560379,29.671290,40.174644,127.215999,12426.62
2,Baseball & Softball,adidas Brazuca 2014 Official Match Ball,10399.350357,27.321370,31.998385,101.280000,10559.95
3,Baseball & Softball,adidas Kids' F5 Messi FG Soccer Cleat,27327.191312,112.164733,95.999155,270.072001,27805.43


### 7. TCO Analysis
TCO dihitung dari biaya pembelian kotor ditambah penalti keterlambatan, kualitas, dan diskon. Karena dataset tidak menyediakan ongkir atau biaya kontrak supplier, komponen TCO dibuat dari variabel yang tersedia pada dataset transaksi.

In [71]:
RI_TABLE = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49,
}

criteria = [
    {"name": "Cost", "column": "tco", "type": "cost"},
    {"name": "Profitability", "column": "avg_profit_margin", "type": "benefit"},
    {"name": "Delivery", "column": "delivery_risk_score", "type": "benefit"},
    {"name": "Quality", "column": "quality_risk_score", "type": "benefit"},
    {"name": "Risk", "column": "risk_score", "type": "benefit"},
    {"name": "Demand", "column": "category_order_share", "type": "benefit"},
    {"name": "Consumer Fit", "column": "consumer_fit_score", "type": "benefit"},
]

criteria_names = [item["name"] for item in criteria]

target_weights = {
    "Cost": 0.18,
    "Profitability": 0.17,
    "Delivery": 0.18,
    "Quality": 0.16,
    "Risk": 0.14,
    "Demand": 0.10,
    "Consumer Fit": 0.07,
}

base_weights = np.array([target_weights[name] for name in criteria_names])
ahp_pairwise_matrix = base_weights[:, None] / base_weights[None, :]

eigenvalues, eigenvectors = np.linalg.eig(ahp_pairwise_matrix)
max_index = np.argmax(eigenvalues.real)
lambda_max = eigenvalues[max_index].real

ahp_weights_array = np.abs(eigenvectors[:, max_index].real)
ahp_weights_array = ahp_weights_array / ahp_weights_array.sum()

n = ahp_pairwise_matrix.shape[0]
ci = (lambda_max - n) / (n - 1)
ri = RI_TABLE[n]
cr = ci / ri
cr = 0.0 if abs(cr) < 1e-12 else cr

ahp_weights = dict(zip(criteria_names, ahp_weights_array.round(6)))

ahp_weight_table = pd.DataFrame(
    {
        "criteria": list(ahp_weights.keys()),
        "weight": list(ahp_weights.values()),
    }
)

print("Consistency Ratio:", round(cr, 6))
print("Consistent:", cr <= 0.10)
ahp_weight_table

Consistency Ratio: 0.0
Consistent: True


,criteria,weight
0,Cost,0.18
1,Profitability,0.17
2,Delivery,0.18
3,Quality,0.16
4,Risk,0.14
5,Demand,0.10
6,Consumer Fit,0.07


### 8. AHP Weighting
AHP dipakai untuk menentukan bobot kriteria. Bobot sudah memasukkan aspek kategori dan konsumen melalui kriteria `Demand` dan `Consumer Fit`. Matrix AHP dibuat konsisten dari bobot target, sehingga bisa langsung dipakai untuk ranking.

In [72]:
def get_matrix(data, criteria_config):
    return data[[item["column"] for item in criteria_config]].astype(float).to_numpy()


def topsis_one_group(data, criteria_config, weights):
    result = data.copy()

    if len(result) == 1:
        result["topsis_score"] = 1.0
        result["topsis_rank"] = 1
        return result

    matrix = get_matrix(result, criteria_config)
    weight_vector = np.array([weights[item["name"]] for item in criteria_config])

    denominator = np.sqrt((matrix ** 2).sum(axis=0))
    denominator[denominator == 0] = 1
    weighted_matrix = (matrix / denominator) * weight_vector

    ideal_best = []
    ideal_worst = []

    for idx, item in enumerate(criteria_config):
        values = weighted_matrix[:, idx]
        if item["type"] == "benefit":
            ideal_best.append(values.max())
            ideal_worst.append(values.min())
        else:
            ideal_best.append(values.min())
            ideal_worst.append(values.max())

    ideal_best = np.array(ideal_best)
    ideal_worst = np.array(ideal_worst)

    distance_best = np.sqrt(((weighted_matrix - ideal_best) ** 2).sum(axis=1))
    distance_worst = np.sqrt(((weighted_matrix - ideal_worst) ** 2).sum(axis=1))
    denominator_score = distance_best + distance_worst

    result["topsis_score"] = np.where(
        denominator_score == 0,
        1.0,
        distance_worst / denominator_score,
    )
    result["topsis_score"] = result["topsis_score"].round(6)
    result["topsis_rank"] = result["topsis_score"].rank(ascending=False, method="dense").astype(int)

    return result


def vikor_one_group(data, criteria_config, weights, v=0.5):
    result = data.copy()

    if len(result) == 1:
        result["vikor_s"] = 0.0
        result["vikor_r"] = 0.0
        result["vikor_q"] = 0.0
        result["vikor_rank"] = 1
        return result

    matrix = get_matrix(result, criteria_config)
    weight_vector = np.array([weights[item["name"]] for item in criteria_config])
    weighted_regret = np.zeros_like(matrix, dtype=float)

    for idx, item in enumerate(criteria_config):
        values = matrix[:, idx]

        if item["type"] == "benefit":
            best = values.max()
            worst = values.min()
            denominator = best - worst
            regret = np.zeros(len(values)) if denominator == 0 else (best - values) / denominator
        else:
            best = values.min()
            worst = values.max()
            denominator = worst - best
            regret = np.zeros(len(values)) if denominator == 0 else (values - best) / denominator

        weighted_regret[:, idx] = regret * weight_vector[idx]

    s_values = weighted_regret.sum(axis=1)
    r_values = weighted_regret.max(axis=1)

    s_best, s_worst = s_values.min(), s_values.max()
    r_best, r_worst = r_values.min(), r_values.max()

    s_term = np.zeros(len(s_values)) if s_worst == s_best else (s_values - s_best) / (s_worst - s_best)
    r_term = np.zeros(len(r_values)) if r_worst == r_best else (r_values - r_best) / (r_worst - r_best)
    q_values = (v * s_term) + ((1 - v) * r_term)

    result["vikor_s"] = s_values.round(6)
    result["vikor_r"] = r_values.round(6)
    result["vikor_q"] = q_values.round(6)
    result["vikor_rank"] = result["vikor_q"].rank(ascending=True, method="dense").astype(int)

    return result


ranked_groups = []

for category_name, group in supplier_df.groupby("category_name", sort=False):
    ranked_group = topsis_one_group(group, criteria, ahp_weights)
    ranked_group = vikor_one_group(ranked_group, criteria, ahp_weights)
    ranked_groups.append(ranked_group)

ranking_df = pd.concat(ranked_groups, ignore_index=True)

ranking_df[
    [
        "category_name",
        "candidate_name",
        "topsis_score",
        "topsis_rank",
        "vikor_q",
        "vikor_rank",
        "risk_score",
        "risk_level",
        "tco",
    ]
].sort_values(["category_name", "topsis_rank"]).head(25)

,category_name,candidate_name,topsis_score,topsis_rank,vikor_q,vikor_rank,risk_score,risk_level,tco
84,Accessories,Team Golf Pittsburgh Steelers Putter Grip,0.886812,1,0.000000,1,4.0761,Low,22540.37
86,Accessories,Team Golf Texas Longhorns Putter Grip,0.658529,2,0.225828,2,3.4931,Medium,22770.64
85,Accessories,Team Golf New England Patriots Putter Grip,0.527975,3,0.617521,4,2.8078,High,20935.77
83,Accessories,Team Golf San Francisco Giants Putter Grip,0.495006,4,0.350525,3,2.9726,High,22155.15
87,Accessories,Team Golf Tennessee Volunteers Putter Grip,0.389612,5,0.865304,5,2.5594,High,23281.83
82,Accessories,Team Golf St. Louis Cardinals Putter Grip,0.266119,6,1.000000,6,2.5053,High,24386.70
35,As Seen on TV!,Nike Men's Free TR 5.0 TB Training Shoe,1.000000,1,0.000000,1,3.0000,Medium,20995.05
101,Baby,Baby sweater,1.000000,1,0.000000,1,3.0000,Medium,12426.62
2,Baseball & Softball,adidas Brazuca 2014 Official Match Ball,0.716238,1,0.000000,1,3.6105,Medium,10559.95
3,Baseball & Softball,adidas Kids' F5 Messi FG Soccer Cleat,0.482701,2,1.000000,3,2.4751,High,27805.43


### 9. TOPSIS dan VIKOR Per Kategori
TOPSIS dan VIKOR dijalankan terpisah untuk setiap kategori. Dengan begitu, ranking alternatif hanya dibandingkan dengan kandidat lain di section yang sama.

In [73]:
ranking_df["average_rank"] = (
    ranking_df["topsis_rank"] + ranking_df["vikor_rank"]
) / 2

ranking_df["final_rank_in_category"] = ranking_df.groupby("category_name")["average_rank"].rank(
    ascending=True,
    method="dense",
).astype(int)


def recommendation(row):
    if not row["prequalified"]:
        return "Rejected - Not Prequalified"
    if not row["compliance_passed"]:
        return "Rejected - Compliance Risk"
    if row["risk_level"] == "Critical":
        return "Not Recommended"
    if row["risk_level"] == "High":
        return "Conditional Supplier"
    if row["final_rank_in_category"] == 1:
        return "Primary Supplier"
    if row["final_rank_in_category"] <= 3:
        return "Backup Supplier"
    return "Not Priority"


ranking_df["recommendation"] = ranking_df.apply(recommendation, axis=1)

final_result = ranking_df.sort_values(
    ["category_name", "final_rank_in_category", "average_rank", "candidate_name"]
).reset_index(drop=True)

final_result[
    [
        "category_name",
        "final_rank_in_category",
        "candidate_id",
        "candidate_name",
        "total_orders",
        "category_order_share",
        "consumer_order_rate",
        "topsis_score",
        "topsis_rank",
        "vikor_q",
        "vikor_rank",
        "risk_score",
        "risk_level",
        "tco",
        "recommendation",
    ]
].head(30)

,category_name,final_rank_in_category,candidate_id,candidate_name,total_orders,category_order_share,consumer_order_rate,topsis_score,topsis_rank,vikor_q,vikor_rank,risk_score,risk_level,tco,recommendation
0,Accessories,1,893,Team Golf Pittsburgh Steelers Putter Grip,293,0.164885,0.508475,0.886812,1,0.000000,1,4.0761,Low,22540.37,Primary Supplier
1,Accessories,2,905,Team Golf Texas Longhorns Putter Grip,298,0.167698,0.511706,0.658529,2,0.225828,2,3.4931,Medium,22770.64,Backup Supplier
2,Accessories,3,897,Team Golf New England Patriots Putter Grip,281,0.158132,0.551601,0.527975,3,0.617521,4,2.8078,High,20935.77,Rejected - Not Prequalified
3,Accessories,3,886,Team Golf San Francisco Giants Putter Grip,292,0.164322,0.506849,0.495006,4,0.350525,3,2.9726,High,22155.15,Rejected - Not Prequalified
4,Accessories,4,906,Team Golf Tennessee Volunteers Putter Grip,300,0.168824,0.443333,0.389612,5,0.865304,5,2.5594,High,23281.83,Conditional Supplier
5,Accessories,5,885,Team Golf St. Louis Cardinals Putter Grip,313,0.176140,0.517572,0.266119,6,1.000000,6,2.5053,High,24386.70,Conditional Supplier
6,As Seen on TV!,1,359,Nike Men's Free TR 5.0 TB Training Shoe,68,1.000000,0.455882,1.000000,1,0.000000,1,3.0000,Medium,20995.05,Primary Supplier
7,Baby,1,1347,Baby sweater,207,1.000000,0.516908,1.000000,1,0.000000,1,3.0000,Medium,12426.62,Primary Supplier
8,Baseball & Softball,1,35,adidas Brazuca 2014 Official Match Ball,65,0.103175,0.507692,0.716238,1,0.000000,1,3.6105,Medium,10559.95,Rejected - Not Prequalified
9,Baseball & Softball,2,37,adidas Kids' F5 Messi FG Soccer Cleat,262,0.415873,0.530534,0.482701,2,1.000000,3,2.4751,High,27805.43,Conditional Supplier


### 10. Final Recommendation Per Kategori
Cell ini menghasilkan rekomendasi akhir di setiap kategori. Output utamanya adalah `final_rank_in_category`, sehingga setiap kategori memiliki Primary Supplier/proxy candidate sendiri.

In [74]:
primary_per_category = (
    final_result[final_result["recommendation"].isin(["Primary Supplier", "Conditional Supplier"])]
    .sort_values(["category_name", "final_rank_in_category", "risk_score"], ascending=[True, True, False])
    .groupby("category_name")
    .head(1)
    .reset_index(drop=True)
)

primary_per_category[
    [
        "category_name",
        "candidate_id",
        "candidate_name",
        "final_rank_in_category",
        "recommendation",
        "risk_level",
        "topsis_score",
        "vikor_q",
        "tco",
        "total_orders",
        "consumer_order_rate",
    ]
].head(30)

,category_name,candidate_id,candidate_name,final_rank_in_category,recommendation,risk_level,topsis_score,vikor_q,tco,total_orders,consumer_order_rate
0,Accessories,893,Team Golf Pittsburgh Steelers Putter Grip,1,Primary Supplier,Low,0.886812,0.000000,22540.37,293,0.508475
1,As Seen on TV!,359,Nike Men's Free TR 5.0 TB Training Shoe,1,Primary Supplier,Medium,1.000000,0.000000,20995.05,68,0.455882
2,Baby,1347,Baby sweater,1,Primary Supplier,Medium,1.000000,0.000000,12426.62,207,0.516908
3,Baseball & Softball,44,adidas Men's F10 Messi TRX FG Soccer Cleat,2,Conditional Supplier,High,0.399031,0.888110,57311.55,303,0.540984
4,Basketball,58,Diamondback Boys' Insight 24 Performance Hybr,1,Primary Supplier,Medium,0.677320,0.010328,8856.33,28,0.379310
5,Books,1346,Sports Books,1,Primary Supplier,Medium,1.000000,0.000000,12802.91,405,0.441975
6,CDs,1348,CDs of rock,1,Primary Supplier,Medium,1.000000,0.000000,3106.33,271,0.553506
7,Cameras,1349,Web Camera,1,Primary Supplier,Medium,1.000000,0.000000,272365.69,592,0.542230
8,Camping & Hiking,957,Diamondback Women's Serene Classic Comfort Bi,1,Primary Supplier,Medium,1.000000,0.000000,4193230.31,12299,0.513002
9,Cardio Equipment,191,Nike Men's Free 5.0+ Running Shoe,1,Primary Supplier,Medium,0.457971,0.000000,3731714.55,11092,0.522393


### 11. Primary Candidate Tiap Kategori
Cell ini merangkum kandidat terbaik untuk setiap kategori. Tabel ini cocok dijadikan output utama untuk sistem supplier selection berbasis kategori.

In [75]:
final_result.to_csv(OUTPUT_DIR / "supplier_selection_by_category_full_result.csv", index=False)
primary_per_category.to_csv(OUTPUT_DIR / "supplier_selection_primary_per_category.csv", index=False)
ahp_weight_table.to_csv(OUTPUT_DIR / "supplier_selection_ahp_weights.csv", index=False)

summary = {
    "total_categories": int(final_result["category_name"].nunique()),
    "total_candidates": int(len(final_result)),
    "prequalified_candidates": int(final_result["prequalified"].sum()),
    "compliance_passed_candidates": int(final_result["compliance_passed"].sum()),
    "ahp_consistency_ratio": float(round(cr, 6)),
    "output_files": [
        "supplier_selection_by_category_full_result.csv",
        "supplier_selection_primary_per_category.csv",
        "supplier_selection_ahp_weights.csv",
    ],
}

with open(OUTPUT_DIR / "supplier_selection_by_category_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

summary

{'total_categories': 50,
 'total_candidates': 118,
 'prequalified_candidates': 89,
 'compliance_passed_candidates': 115,
 'ahp_consistency_ratio': 0.0,
 'output_files': ['supplier_selection_by_category_full_result.csv',
  'supplier_selection_primary_per_category.csv',
  'supplier_selection_ahp_weights.csv']}

### 12. Export Output
Cell ini menyimpan hasil akhir ke CSV dan JSON. File utama adalah `supplier_selection_primary_per_category.csv` untuk rekomendasi utama per kategori dan `supplier_selection_by_category_full_result.csv` untuk hasil lengkap seluruh kandidat.

### 13. Tes Kode Berjalan: Ranking Berdasarkan Input Kategori
Cell di bawah ini digunakan untuk menguji hasil supplier selection. User memasukkan salah satu `category_name`, lalu sistem menampilkan ranking kandidat hanya untuk kategori tersebut.

In [76]:
if "final_result" not in globals():
    raise RuntimeError("Jalankan semua cell sebelumnya terlebih dahulu agar final_result tersedia.")

available_categories = sorted(final_result["category_name"].dropna().unique())

print("Contoh category yang tersedia:")
for category in available_categories[:15]:
    print("-", category)

category_input = input("\nMasukkan Category Name yang ingin diranking: ").strip()

if not category_input:
    raise ValueError("Category Name tidak boleh kosong.")

exact_matches = [
    category for category in available_categories
    if category.lower() == category_input.lower()
]

partial_matches = [
    category for category in available_categories
    if category_input.lower() in category.lower()
]

if exact_matches:
    selected_category = exact_matches[0]
elif len(partial_matches) == 1:
    selected_category = partial_matches[0]
    print(f"\nKategori tidak exact, memakai hasil terdekat: {selected_category}")
else:
    print("\nCategory tidak ditemukan atau terlalu ambigu.")
    if partial_matches:
        print("Kategori yang mirip:")
        for category in partial_matches[:20]:
            print("-", category)
    else:
        print("Tidak ada kategori yang mirip. Cek kembali penulisan category_name.")
    raise ValueError("Masukkan Category Name yang lebih spesifik atau sesuai daftar kategori.")

category_ranking = (
    final_result[final_result["category_name"] == selected_category]
    .sort_values(["final_rank_in_category", "average_rank", "candidate_name"])
    .reset_index(drop=True)
)

display_columns = [
    "category_name",
    "final_rank_in_category",
    "candidate_id",
    "candidate_name",
    "total_orders",
    "category_order_share",
    "consumer_order_rate",
    "topsis_score",
    "topsis_rank",
    "vikor_q",
    "vikor_rank",
    "risk_score",
    "risk_level",
    "tco",
    "recommendation",
]

print(f"\nRanking kandidat untuk kategori: {selected_category}")
print(f"Jumlah kandidat: {len(category_ranking)}")

category_ranking[display_columns]


Contoh category yang tersedia:
- Accessories
- As Seen on  TV!
- Baby 
- Baseball & Softball
- Basketball
- Books 
- Boxing & MMA
- CDs 
- Cameras 
- Camping & Hiking
- Cardio Equipment
- Children's Clothing
- Cleats
- Computers
- Consumer Electronics

Ranking kandidat untuk kategori: Accessories
Jumlah kandidat: 6


,category_name,final_rank_in_category,candidate_id,candidate_name,total_orders,category_order_share,consumer_order_rate,topsis_score,topsis_rank,vikor_q,vikor_rank,risk_score,risk_level,tco,recommendation
0,Accessories,1,893,Team Golf Pittsburgh Steelers Putter Grip,293,0.164885,0.508475,0.886812,1,0.000000,1,4.0761,Low,22540.37,Primary Supplier
1,Accessories,2,905,Team Golf Texas Longhorns Putter Grip,298,0.167698,0.511706,0.658529,2,0.225828,2,3.4931,Medium,22770.64,Backup Supplier
2,Accessories,3,897,Team Golf New England Patriots Putter Grip,281,0.158132,0.551601,0.527975,3,0.617521,4,2.8078,High,20935.77,Rejected - Not Prequalified
3,Accessories,3,886,Team Golf San Francisco Giants Putter Grip,292,0.164322,0.506849,0.495006,4,0.350525,3,2.9726,High,22155.15,Rejected - Not Prequalified
4,Accessories,4,906,Team Golf Tennessee Volunteers Putter Grip,300,0.168824,0.443333,0.389612,5,0.865304,5,2.5594,High,23281.83,Conditional Supplier
5,Accessories,5,885,Team Golf St. Louis Cardinals Putter Grip,313,0.176140,0.517572,0.266119,6,1.000000,6,2.5053,High,24386.70,Conditional Supplier
